[Reference](https://medium.com/@sebuzdugan/timber-is-ollama-for-the-models-you-actually-run-in-production-79df9f3897bc)

# Step 1: Train and export your model

In [1]:
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
dtrain = xgb.DMatrix(X_train, label=y_train)
params = {
    "objective": "binary:logistic",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "eval_metric": "logloss",
}
bst = xgb.train(params, dtrain, num_boost_round=200)
# Export to JSON for Timber
bst.save_model("model.json")

# Step 2: Compile with Timber
```
timber build \
  --model model.json \
  --format xgboost_json \
  --out model_bin
```

# Step 3: Serve via HTTP
```
timber serve \
  --model-binary ./model_bin \
  --host 0.0.0.0 \
  --port 8080
```

# Step 4: Call it from your app
```
curl -X POST http://localhost:8080/predict \
  -H "Content-Type: application/json" \
  -d '{
    "features": {
      "mean_radius": 14.1,
      "mean_texture": 20.9,
      "mean_perimeter": 92.8,
      "mean_area": 600.1
      // ... other features ...
    }
  }'
```